# Card Transaction Bronze Generator

This notebook generates synthetic bronze-layer card transaction data in Unity Catalog for fraud analytics and feature engineering. It creates customer and merchant dimensions, transaction facts with seeded fraud patterns, prior fraud cases, and validation outputs with reproducible random seeds.

In [0]:
%uv pip install Faker

Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-7c26aa15-da87-40c0-87d8-3a742fe30da4
Resolved 1 package in 144ms
 Downloaded faker
Prepared 1 package in 170ms
Installed 1 package in 76ms
 + faker==40.41.0
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import DoubleType, StringType
from pyspark.sql.functions import pandas_udf
import pandas as pd

SEED = 42
CATALOG = "workshop_guy_catalog"
SCHEMA = "fe-bar-a0"
CUSTOMER_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`dim_customer`"
MERCHANT_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`dim_merchant`"
TRANSACTION_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`fact_transactions`"
CASE_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`fact_prior_fraud_cases`"
SCHEMA_FQN = f"`{CATALOG}`.`{SCHEMA}`"

N_CUSTOMERS = 10_000
N_MERCHANTS = 5_000
N_TRANSACTIONS = 500_000
N_CASES = 50_000
DIM_PARTITIONS = 16
FACT_PARTITIONS = 32

HOME_REGION_WEIGHTS = [
    ("US-East", 0.30),
    ("US-West", 0.25),
    ("EU", 0.20),
    ("APAC", 0.15),
    ("LATAM", 0.10),
]
RISK_SEGMENT_WEIGHTS = [
    ("low", 0.60),
    ("medium", 0.25),
    ("high", 0.15),
]
MCC_WEIGHTS = [
    ("retail", 0.30),
    ("dining", 0.20),
    ("travel", 0.15),
    ("online_services", 0.15),
    ("groceries", 0.10),
    ("entertainment", 0.10),
]
MERCHANT_RISK_WEIGHTS = [
    ("low", 0.50),
    ("medium", 0.30),
    ("high", 0.20),
]
COUNTRY_WEIGHTS = [
    ("US", 0.40),
    ("UK", 0.15),
    ("DE", 0.10),
    ("JP", 0.08),
    ("BR", 0.07),
    ("IN", 0.05),
    ("CA", 0.05),
    ("MX", 0.04),
    ("FR", 0.03),
    ("SG", 0.03),
]
CHANNEL_WEIGHTS = [
    ("card_present", 0.65),
    ("online", 0.35),
]
FRAUD_TYPE_WEIGHTS = [
    ("account_takeover", 0.25),
    ("card_not_present", 0.25),
    ("counterfeit", 0.20),
    ("friendly_fraud", 0.15),
    ("identity_theft", 0.15),
]


def weighted_choice(rand_col, weighted_labels):
    expr = None
    cumulative = 0.0
    for label, weight in weighted_labels[:-1]:
        cumulative += weight
        expr = F.when(rand_col < F.lit(cumulative), F.lit(label)) if expr is None else expr.when(rand_col < F.lit(cumulative), F.lit(label))
    return expr.otherwise(F.lit(weighted_labels[-1][0]))


def region_to_country(region_col):
    return (
        F.when(region_col.isin("US-East", "US-West"), F.lit("US"))
         .when(region_col == F.lit("EU"), F.lit("UK"))
         .when(region_col == F.lit("APAC"), F.lit("JP"))
         .otherwise(F.lit("BR"))
    )


@pandas_udf(DoubleType())
def faker_spend_bias(id_series: pd.Series) -> pd.Series:
    from faker import Faker
    values = []
    for raw_id in id_series.fillna(0).astype(int).tolist():
        fake = Faker()
        fake.seed_instance(SEED + int(raw_id))
        values.append(fake.random_int(min=94, max=112) / 100.0)
    return pd.Series(values)


@pandas_udf(StringType())
def faker_device_token(key_series: pd.Series) -> pd.Series:
    from faker import Faker
    values = []
    for raw_key in key_series.fillna(0).astype(int).tolist():
        fake = Faker()
        fake.seed_instance(SEED + int(raw_key))
        values.append(fake.lexify(text="?????").upper())
    return pd.Series(values)


print(
    {
        "catalog": CATALOG,
        "schema": SCHEMA,
        "rows": {
            "dim_customer": N_CUSTOMERS,
            "dim_merchant": N_MERCHANTS,
            "fact_transactions": N_TRANSACTIONS,
            "fact_prior_fraud_cases": N_CASES,
        },
        "seed": SEED,
        "partitions": {"dimension": DIM_PARTITIONS, "fact": FACT_PARTITIONS},
    }
)

{'catalog': 'workshop_guy_catalog', 'schema': 'fe-bar-a0', 'rows': {'dim_customer': 10000, 'dim_merchant': 5000, 'fact_transactions': 500000, 'fact_prior_fraud_cases': 50000}, 'seed': 42, 'partitions': {'dimension': 16, 'fact': 32}}


In [0]:
current_user = spark.sql("SELECT current_user() AS current_user").first()["current_user"]
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_FQN}")

describe_schema_df = spark.sql(f"DESCRIBE SCHEMA EXTENDED {SCHEMA_FQN}")
display(describe_schema_df)

owner = current_user

grant_df = spark.sql(f"SHOW GRANTS ON SCHEMA {SCHEMA_FQN}")
principal_col = next((c for c in grant_df.columns if c.lower() in {"principal", "grantee"}), grant_df.columns[0])
principals = [row["principal"] for row in grant_df.select(F.col(principal_col).alias("principal")).distinct().collect()]
principals_to_restrict = sorted({p for p in principals if p and p not in {current_user, owner}})

if principals_to_restrict:
    print(
        {
            "warning": "Schema permission lockdown still requires manual REVOKE approval outside this notebook run.",
            "pending_non_owner_grants": principals_to_restrict,
        }
    )

print({"current_user": current_user, "owner": owner, "non_owner_grants": principals_to_restrict})
final_grants_df = spark.sql(f"SHOW GRANTS ON SCHEMA {SCHEMA_FQN}")
display(final_grants_df)

database_description_item,database_description_value
Catalog Name,workshop_guy_catalog
Namespace Name,`fe-bar-a0`
Comment,
Location,
Owner,alex.owen@databricks.com
Properties,"((unity.catalog.managed.delta.defaults.delta.parquet.format.version,2.12.0), (unity.catalog.managed.delta.defaults.delta.parquet.format.version.afe.internal,2.12.0), (unity.catalog.managed.iceberg.defaults.delta.feature.catalogManaged,supported))"
Predictive Optimization,ENABLE (inherited from METASTORE metastore_aws_us_east_1)


{'warning': 'Schema permission lockdown still requires manual REVOKE approval outside this notebook run.', 'pending_non_owner_grants': ['account users', 'suraj.bang@databricks.com']}
{'current_user': 'alex.owen@databricks.com', 'owner': 'alex.owen@databricks.com', 'non_owner_grants': ['account users', 'suraj.bang@databricks.com']}


Principal,ActionType,ObjectType,ObjectKey
account users,ALL PRIVILEGES,CATALOG,workshop_guy_catalog
account users,EXTERNAL USE SCHEMA,CATALOG,workshop_guy_catalog
account users,MANAGE,CATALOG,workshop_guy_catalog
alex.owen@databricks.com,ALL PRIVILEGES,CATALOG,workshop_guy_catalog
alex.owen@databricks.com,MANAGE,CATALOG,workshop_guy_catalog
suraj.bang@databricks.com,READ METADATA,METASTORE,metastore_aws_us_east_1


In [0]:
customer_df = (
    spark.range(0, N_CUSTOMERS, 1, DIM_PARTITIONS)
    .withColumn("customer_id", F.format_string("CUST-%05d", F.col("id") + F.lit(1)))
    .withColumn("region_roll", F.rand(SEED + 1))
    .withColumn("risk_roll", F.rand(SEED + 2))
    .withColumn("tenure_roll", F.rand(SEED + 3))
    .withColumn("new_tenure_roll", F.rand(SEED + 4))
    .withColumn("core_tenure_roll", F.rand(SEED + 5))
    .withColumn("mature_tenure_roll", F.rand(SEED + 6))
    .withColumn("home_region", weighted_choice(F.col("region_roll"), HOME_REGION_WEIGHTS))
    .withColumn("risk_segment", weighted_choice(F.col("risk_roll"), RISK_SEGMENT_WEIGHTS))
    .withColumn(
        "tenure",
        F.when(F.col("tenure_roll") < 0.12, (F.floor(F.col("new_tenure_roll") * 3) + 1).cast("int"))
         .when(F.col("tenure_roll") < 0.87, (F.floor(F.col("core_tenure_roll") * 49) + 12).cast("int"))
         .otherwise((F.floor(F.col("mature_tenure_roll") * 120) + 61).cast("int")),
    )
    .withColumn(
        "base_median_spend",
        F.when(F.col("risk_segment") == "low", F.lit(500.0))
         .when(F.col("risk_segment") == "medium", F.lit(1500.0))
         .otherwise(F.lit(3000.0)),
    )
    .withColumn(
        "spend_sigma",
        F.when(F.col("risk_segment") == "low", F.lit(0.48))
         .when(F.col("risk_segment") == "medium", F.lit(0.52))
         .otherwise(F.lit(0.58)),
    )
    .withColumn("faker_spend_bias", faker_spend_bias(F.col("id")))
    .withColumn(
        "avg_monthly_spend",
        F.round(
            F.greatest(
                F.exp(F.log(F.col("base_median_spend")) + F.col("spend_sigma") * F.randn(SEED + 7))
                * F.col("faker_spend_bias")
                * (F.lit(1.0) + F.least(F.col("tenure"), F.lit(120)) / F.lit(2400.0)),
                F.lit(50.0),
            ),
            2,
        ),
    )
    .select("customer_id", "tenure", "home_region", "risk_segment", "avg_monthly_spend")
)

customer_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(CUSTOMER_TABLE)
spark.sql(
    f"COMMENT ON TABLE {CUSTOMER_TABLE} IS 'Synthetic customer dimension for fraud analytics with weighted region, risk, tenure, and spend distributions.'"
)

customer_table_df = spark.table(CUSTOMER_TABLE)
print({"table": CUSTOMER_TABLE, "row_count": customer_table_df.count()})
display(customer_table_df.orderBy("customer_id").limit(5))
display(
    customer_table_df.groupBy("home_region", "risk_segment")
    .count()
    .orderBy("home_region", "risk_segment")
)
display(customer_table_df.agg(
    F.round(F.avg("avg_monthly_spend"), 2).alias("avg_spend"),
    F.expr("percentile_approx(avg_monthly_spend, array(0.1, 0.5, 0.9))").alias("spend_percentiles"),
    F.round(F.avg("tenure"), 2).alias("avg_tenure_months")
))

{'table': '`workshop_guy_catalog`.`fe-bar-a0`.`dim_customer`', 'row_count': 10000}


customer_id,tenure,home_region,risk_segment,avg_monthly_spend
CUST-00001,45,EU,medium,524.66
CUST-00002,43,EU,low,752.77
CUST-00003,25,EU,low,656.6
CUST-00004,33,US-East,low,491.37
CUST-00005,33,APAC,low,435.8


home_region,risk_segment,count
APAC,high,226
APAC,low,863
APAC,medium,406
EU,high,320
EU,low,1207
EU,medium,508
LATAM,high,153
LATAM,low,591
LATAM,medium,264
US-East,high,438


avg_spend,spend_percentiles,avg_tenure_months
1357.47,"List(325.25, 772.8, 3142.97)",43.65


In [0]:
merchant_df = (
    spark.range(0, N_MERCHANTS, 1, DIM_PARTITIONS)
    .withColumn("merchant_id", F.format_string("MERCH-%05d", F.col("id") + F.lit(1)))
    .withColumn("mcc_roll", F.rand(SEED + 11))
    .withColumn("risk_roll", F.rand(SEED + 12))
    .withColumn("country_roll", F.rand(SEED + 13))
    .withColumn("mcc_category", weighted_choice(F.col("mcc_roll"), MCC_WEIGHTS))
    .withColumn("risk_tier", weighted_choice(F.col("risk_roll"), MERCHANT_RISK_WEIGHTS))
    .withColumn("country", weighted_choice(F.col("country_roll"), COUNTRY_WEIGHTS))
    .select("merchant_id", "mcc_category", "risk_tier", "country")
)

merchant_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(MERCHANT_TABLE)
spark.sql(
    f"COMMENT ON TABLE {MERCHANT_TABLE} IS 'Synthetic merchant dimension with weighted MCC categories, risk tiers, and countries for cross-border fraud analysis.'"
)

merchant_table_df = spark.table(MERCHANT_TABLE)
print({"table": MERCHANT_TABLE, "row_count": merchant_table_df.count()})
display(merchant_table_df.orderBy("merchant_id").limit(5))
display(
    merchant_table_df.groupBy("mcc_category", "risk_tier")
    .count()
    .orderBy("mcc_category", "risk_tier")
)
display(merchant_table_df.groupBy("country").count().orderBy(F.desc("count"), "country"))

{'table': '`workshop_guy_catalog`.`fe-bar-a0`.`dim_merchant`', 'row_count': 5000}


merchant_id,mcc_category,risk_tier,country
MERCH-00001,retail,medium,US
MERCH-00002,dining,medium,US
MERCH-00003,entertainment,low,SG
MERCH-00004,travel,low,US
MERCH-00005,travel,low,UK


mcc_category,risk_tier,count
dining,high,213
dining,low,513
dining,medium,273
entertainment,high,96
entertainment,low,260
entertainment,medium,142
groceries,high,96
groceries,low,229
groceries,medium,149
online_services,high,148


country,count
US,2029
UK,737
DE,495
JP,402
BR,363
CA,239
IN,231
MX,195
SG,159
FR,150


In [0]:
customer_lookup = (
    spark.table(CUSTOMER_TABLE)
    .withColumn("customer_idx", F.regexp_extract("customer_id", "(\\d+)$", 1).cast("int") - F.lit(1))
)
merchant_lookup = (
    spark.table(MERCHANT_TABLE)
    .withColumn("merchant_idx", F.regexp_extract("merchant_id", "(\\d+)$", 1).cast("int") - F.lit(1))
)

base_transactions = (
    spark.range(0, N_TRANSACTIONS, 1, FACT_PARTITIONS)
    .withColumn("txn_id", F.format_string("TXN-%07d", F.col("id") + F.lit(1)))
    .withColumn("customer_idx", F.pmod(F.xxhash64(F.col("id"), F.lit(SEED)), F.lit(N_CUSTOMERS)).cast("int"))
    .withColumn("merchant_idx", F.pmod(F.xxhash64(F.col("id"), F.lit(SEED + 1)), F.lit(N_MERCHANTS)).cast("int"))
    .withColumn("channel_roll", F.rand(SEED + 14))
    .withColumn("days_roll", F.rand(SEED + 15))
    .withColumn("channel", weighted_choice(F.col("channel_roll"), CHANNEL_WEIGHTS))
    .withColumn("days_back", F.floor(F.pow(F.col("days_roll"), F.lit(1.22)) * F.lit(180)).cast("int"))
    .withColumn("txn_date", F.date_sub(F.current_date(), F.col("days_back")))
    .drop("channel_roll", "days_roll")
)

hour_rand = F.rand(SEED + 16)
minute_rand = F.rand(SEED + 17)
second_rand = F.rand(SEED + 18)
card_present_hour = (
    F.when(hour_rand < 0.08, (F.floor(F.rand(SEED + 101) * 2) + 6).cast("int"))
     .when(hour_rand < 0.28, (F.floor(F.rand(SEED + 102) * 3) + 8).cast("int"))
     .when(hour_rand < 0.55, (F.floor(F.rand(SEED + 103) * 4) + 11).cast("int"))
     .when(hour_rand < 0.82, (F.floor(F.rand(SEED + 104) * 4) + 15).cast("int"))
     .when(hour_rand < 0.97, (F.floor(F.rand(SEED + 105) * 3) + 19).cast("int"))
     .otherwise(F.lit(22))
)
online_hour = (
    F.when(hour_rand < 0.10, (F.floor(F.rand(SEED + 106) * 6)).cast("int"))
     .when(hour_rand < 0.30, (F.floor(F.rand(SEED + 107) * 5) + 6).cast("int"))
     .when(hour_rand < 0.65, (F.floor(F.rand(SEED + 108) * 6) + 11).cast("int"))
     .when(hour_rand < 0.90, (F.floor(F.rand(SEED + 109) * 5) + 17).cast("int"))
     .otherwise((F.floor(F.rand(SEED + 110) * 2) + 22).cast("int"))
)

transaction_df = (
    base_transactions
    .join(customer_lookup, on="customer_idx", how="inner")
    .join(merchant_lookup, on="merchant_idx", how="inner")
    .withColumn(
        "timestamp",
        F.make_timestamp(
            F.year("txn_date"),
            F.month("txn_date"),
            F.dayofmonth("txn_date"),
            F.when(F.col("channel") == "card_present", card_present_hour).otherwise(online_hour),
            F.floor(minute_rand * 60).cast("int"),
            F.floor(second_rand * 60).cast("int"),
        ),
    )
    .withColumn("home_country", region_to_country(F.col("home_region")))
    .withColumn("cross_border", F.col("country") != F.col("home_country"))
    .withColumn(
        "mcc_multiplier",
        F.when(F.col("mcc_category") == "travel", F.lit(3.8))
         .when(F.col("mcc_category") == "retail", F.lit(1.55))
         .when(F.col("mcc_category") == "entertainment", F.lit(1.70))
         .when(F.col("mcc_category") == "dining", F.lit(1.05))
         .when(F.col("mcc_category") == "groceries", F.lit(0.72))
         .otherwise(F.lit(0.88)),
    )
    .withColumn(
        "risk_multiplier",
        F.when(F.col("risk_tier") == "high", F.lit(1.22))
         .when(F.col("risk_tier") == "medium", F.lit(1.08))
         .otherwise(F.lit(1.0)),
    )
    .withColumn("channel_multiplier", F.when(F.col("channel") == "online", F.lit(1.14)).otherwise(F.lit(1.0)))
    .withColumn(
        "base_amount",
        F.exp(F.lit(3.91202300543) + F.lit(0.92) * F.randn(SEED + 19))
        * F.col("mcc_multiplier")
        * F.col("risk_multiplier")
        * F.col("channel_multiplier"),
    )
    .withColumn("seed_high_amount", F.rand(SEED + 20) < 0.006)
    .withColumn("seed_unusual_spend", F.rand(SEED + 21) < 0.010)
    .withColumn(
        "amount",
        F.round(
            F.greatest(
                F.when(F.col("seed_high_amount"), F.greatest(F.col("base_amount"), F.lit(2200.0) + F.abs(F.randn(SEED + 22)) * F.lit(2800.0)))
                 .otherwise(F.col("base_amount")),
                F.when(F.col("seed_unusual_spend"), F.col("avg_monthly_spend") * (F.lit(3.25) + F.rand(SEED + 23) * F.lit(2.35)))
                 .otherwise(F.col("base_amount")),
                F.lit(1.0),
            ),
            2,
        ),
    )
    .withColumn(
        "home_device_slots",
        F.when(F.pmod(F.xxhash64(F.col("customer_id"), F.lit(SEED + 24)), F.lit(100)) < 5, F.lit(1))
         .when(F.pmod(F.xxhash64(F.col("customer_id"), F.lit(SEED + 24)), F.lit(100)) < 15, F.lit(2))
         .when(F.pmod(F.xxhash64(F.col("customer_id"), F.lit(SEED + 24)), F.lit(100)) < 35, F.lit(3))
         .when(F.pmod(F.xxhash64(F.col("customer_id"), F.lit(SEED + 24)), F.lit(100)) < 60, F.lit(4))
         .when(F.pmod(F.xxhash64(F.col("customer_id"), F.lit(SEED + 24)), F.lit(100)) < 85, F.lit(5))
         .otherwise(F.lit(6)),
    )
    .withColumn(
        "device_slot",
        F.when(
            (F.col("channel") == "online") & (F.rand(SEED + 25) < 0.06),
            (F.pmod(F.xxhash64(F.col("txn_id"), F.lit(SEED + 26)), F.lit(4)) + F.lit(7)).cast("int"),
        ).otherwise(
            (F.pmod(F.xxhash64(F.col("txn_id"), F.lit(SEED + 27)), F.col("home_device_slots")) + F.lit(1)).cast("int")
        ),
    )
)

customer_window = Window.partitionBy("customer_id").orderBy("timestamp", "txn_id")
device_window = Window.partitionBy("customer_id", "device_slot")

transaction_df = (
    transaction_df
    .withColumn("customer_txn_seq", F.row_number().over(customer_window))
    .withColumn("first_seen_seq", F.min("customer_txn_seq").over(device_window))
    .withColumn("is_new_device", F.col("customer_txn_seq") == F.col("first_seen_seq"))
    .withColumn("device_numeric", F.col("customer_idx") * F.lit(10) + F.col("device_slot"))
    .withColumn(
        "device_id",
        F.concat(
            F.lit("DEV-"),
            faker_device_token(F.col("device_numeric")),
            F.lit("-"),
            F.lpad(F.col("device_numeric").cast("string"), 6, "0"),
        ),
    )
    .withColumn("signal_high_amount_crossborder", (F.col("amount") > 2000) & F.col("cross_border"))
    .withColumn("signal_new_device_online", F.col("is_new_device") & (F.col("channel") == "online"))
    .withColumn("signal_unusual_spend", F.col("amount") > (F.col("avg_monthly_spend") * 3))
    .withColumn(
        "signal_crossborder_highrisk_combo",
        F.col("cross_border") & (F.col("risk_tier") == "high") & (F.col("risk_segment") == "high"),
    )
    .withColumn(
        "signal_count",
        F.col("signal_high_amount_crossborder").cast("int")
        + F.col("signal_new_device_online").cast("int")
        + F.col("signal_unusual_spend").cast("int")
        + F.col("signal_crossborder_highrisk_combo").cast("int"),
    )
    .withColumn(
        "signal_base_prob",
        F.greatest(
            F.when(F.col("signal_high_amount_crossborder"), F.lit(0.25)).otherwise(F.lit(0.0)),
            F.when(F.col("signal_new_device_online"), F.lit(0.15)).otherwise(F.lit(0.0)),
            F.when(F.col("signal_unusual_spend"), F.lit(0.20)).otherwise(F.lit(0.0)),
            F.when(F.col("signal_crossborder_highrisk_combo"), F.lit(0.30)).otherwise(F.lit(0.0)),
        ),
    )
    .withColumn(
        "fraud_prob",
        F.least(
            F.lit(0.005)
            + F.col("signal_base_prob")
            + F.when(F.col("signal_count") > 1, (F.col("signal_count") - 1) * F.lit(0.02)).otherwise(F.lit(0.0))
            + F.when((F.col("channel") == "online") & (F.col("signal_count") > 0), F.lit(0.005)).otherwise(F.lit(0.0)),
            F.lit(0.34),
        ),
    )
    .withColumn("is_fraud", F.rand(SEED + 28) < F.col("fraud_prob"))
    .withColumn(
        "geo_distance_from_home",
        F.round(
            F.when(
                F.col("cross_border"),
                F.greatest(F.exp(F.lit(2.35) + F.lit(0.55) * F.randn(SEED + 29)), F.lit(450.0) + F.abs(F.randn(SEED + 30)) * F.lit(320.0)),
            )
             .when(
                F.col("is_fraud") & F.col("signal_new_device_online"),
                F.greatest(F.exp(F.lit(3.15) + F.lit(0.55) * F.randn(SEED + 31)), F.lit(120.0) + F.abs(F.randn(SEED + 32)) * F.lit(140.0)),
             )
             .otherwise(F.exp(F.lit(2.25) + F.lit(0.62) * F.randn(SEED + 33))),
            2,
        ),
    )
    .select(
        "txn_id",
        "customer_id",
        "merchant_id",
        "amount",
        "timestamp",
        "channel",
        "device_id",
        "geo_distance_from_home",
        "is_fraud",
    )
)

transaction_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TRANSACTION_TABLE)
spark.sql(
    f"COMMENT ON TABLE {TRANSACTION_TABLE} IS 'Synthetic card transactions over the last six months with probabilistic fraud seeded from high-risk behavioral and cross-border patterns.'"
)

transaction_table_df = spark.table(TRANSACTION_TABLE)
print({
    "table": TRANSACTION_TABLE,
    "row_count": transaction_table_df.count(),
    "fraud_rate_pct": transaction_table_df.agg(F.round(F.avg(F.col("is_fraud").cast("double")) * 100, 2).alias("fraud_rate_pct")).first()["fraud_rate_pct"],
})
display(transaction_table_df.orderBy("timestamp").limit(5))
display(transaction_table_df.groupBy("channel", "is_fraud").count().orderBy("channel", "is_fraud"))
display(transaction_table_df.agg(
    F.round(F.avg("amount"), 2).alias("avg_amount"),
    F.expr("percentile_approx(amount, array(0.5, 0.9, 0.99))").alias("amount_percentiles"),
    F.expr("approx_count_distinct(device_id)").alias("approx_distinct_devices")
))

{'table': '`workshop_guy_catalog`.`fe-bar-a0`.`fact_transactions`', 'row_count': 500000, 'fraud_rate_pct': 2.3}


txn_id,customer_id,merchant_id,amount,timestamp,channel,device_id,geo_distance_from_home,is_fraud
TXN-0239739,CUST-08419,MERCH-04742,18.56,2026-04-12T00:06:09.000Z,online,DEV-XIKTY-084183,518.53,false
TXN-0230881,CUST-00831,MERCH-01524,155.62,2026-04-12T00:07:45.000Z,online,DEV-YATXQ-008302,737.6,true
TXN-0454772,CUST-07213,MERCH-00867,197.37,2026-04-12T00:12:06.000Z,online,DEV-NFIWP-072124,493.65,false
TXN-0245246,CUST-04400,MERCH-02176,131.13,2026-04-12T00:13:12.000Z,online,DEV-MQFJH-043992,13.22,false
TXN-0094385,CUST-07201,MERCH-02033,17.12,2026-04-12T00:15:10.000Z,online,DEV-KLWCL-072002,462.67,false


channel,is_fraud,count
card_present,false,319914
card_present,true,5209
online,false,168608
online,true,6269


avg_amount,amount_percentiles,approx_distinct_devices
218.36,"List(78.4, 333.94, 3082.87)",48421


In [0]:
transactions_for_cases = spark.table(TRANSACTION_TABLE)
fraud_pool = transactions_for_cases.filter(F.col("is_fraud")).withColumn("pool_rank", F.row_number().over(Window.orderBy("txn_id")))
nonfraud_pool = transactions_for_cases.filter(~F.col("is_fraud")).withColumn("pool_rank", F.row_number().over(Window.orderBy("txn_id")))

fraud_count = fraud_pool.count()
nonfraud_count = nonfraud_pool.count()
if fraud_count == 0:
    raise ValueError("Fraud pool is empty; adjust fraud seeding before generating prior fraud cases.")
if nonfraud_count < 20_000:
    raise ValueError("Not enough non-fraud transactions to build the requested prior case sample.")

fraud_case_source = (
    spark.range(0, 30_000, 1, DIM_PARTITIONS)
    .withColumn("pool_rank", (F.pmod(F.col("id"), F.lit(fraud_count)) + F.lit(1)).cast("int"))
    .join(fraud_pool.select("pool_rank", "txn_id", "customer_id", "amount", "timestamp"), on="pool_rank", how="inner")
    .withColumn("source_type", F.lit("fraud_pool"))
    .drop("pool_rank")
)
nonfraud_case_source = (
    spark.range(0, 20_000, 1, DIM_PARTITIONS)
    .withColumn("pool_rank", (F.col("id") + F.lit(1)).cast("int"))
    .join(nonfraud_pool.select("pool_rank", "txn_id", "customer_id", "amount", "timestamp"), on="pool_rank", how="inner")
    .withColumn("source_type", F.lit("nonfraud_pool"))
    .drop("pool_rank")
)

case_source = fraud_case_source.unionByName(nonfraud_case_source)
case_window = Window.orderBy("source_type", "txn_id", "customer_id")

case_df = (
    case_source
    .withColumn("case_seq", F.row_number().over(case_window))
    .withColumn("case_id", F.format_string("CASE-%07d", F.col("case_seq")))
    .withColumn(
        "investigation_outcome",
        F.when(
            F.col("source_type") == "fraud_pool",
            F.when(F.rand(SEED + 41) < 0.90, F.lit("confirmed_fraud"))
             .when(F.rand(SEED + 41) < 0.95, F.lit("false_positive"))
             .otherwise(F.lit("suspicious_not_confirmed")),
        ).otherwise(
            F.when(F.rand(SEED + 42) < 0.15, F.lit("confirmed_fraud"))
             .when(F.rand(SEED + 42) < 0.70, F.lit("false_positive"))
             .otherwise(F.lit("suspicious_not_confirmed")),
        ),
    )
    .withColumn(
        "fraud_type",
        F.when(F.col("investigation_outcome") == "confirmed_fraud", weighted_choice(F.rand(SEED + 43), FRAUD_TYPE_WEIGHTS))
         .otherwise(F.lit(None).cast("string")),
    )
    .withColumn("resolution_days", (F.floor(F.rand(SEED + 44) * 30) + 1).cast("int"))
    .withColumn(
        "resolution_date",
        F.least(
            F.current_timestamp(),
            F.from_unixtime(
                F.unix_timestamp("timestamp")
                + F.col("resolution_days") * F.lit(86400)
                + F.pmod(F.xxhash64(F.col("case_id"), F.lit(SEED + 45)), F.lit(86400))
            ).cast("timestamp"),
        ),
    )
    .withColumn(
        "recovery_amount",
        F.round(
            F.when(
                F.col("investigation_outcome") == "confirmed_fraud",
                F.col("amount") * (F.lit(0.25) + F.rand(SEED + 46) * F.lit(0.70)),
            )
             .when(
                F.col("investigation_outcome") == "suspicious_not_confirmed",
                F.col("amount") * (F.rand(SEED + 47) * F.lit(0.15)),
             )
             .otherwise(F.lit(0.0)),
            2,
        ),
    )
    .select(
        "case_id",
        "txn_id",
        "customer_id",
        "investigation_outcome",
        "resolution_date",
        "fraud_type",
        "recovery_amount",
    )
)

case_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(CASE_TABLE)
spark.sql(
    f"COMMENT ON TABLE {CASE_TABLE} IS 'Synthetic prior fraud investigations linked to transactions for model training, analyst workflows, and grounding corpora.'"
)

case_table_df = spark.table(CASE_TABLE)
print({"table": CASE_TABLE, "row_count": case_table_df.count()})
display(case_table_df.orderBy("case_id").limit(5))
display(case_table_df.groupBy("investigation_outcome").count().orderBy("investigation_outcome"))
display(case_table_df.agg(
    F.round(F.avg("recovery_amount"), 2).alias("avg_recovery_amount"),
    F.expr("percentile_approx(recovery_amount, array(0.5, 0.9, 0.99))").alias("recovery_percentiles")
))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


{'table': '`workshop_guy_catalog`.`fe-bar-a0`.`fact_prior_fraud_cases`', 'row_count': 50000}


case_id,txn_id,customer_id,investigation_outcome,resolution_date,fraud_type,recovery_amount
CASE-0000001,TXN-0000007,CUST-06772,confirmed_fraud,2026-10-06T19:51:53.000Z,account_takeover,5932.7
CASE-0000002,TXN-0000007,CUST-06772,confirmed_fraud,2026-09-30T22:05:19.000Z,card_not_present,10365.33
CASE-0000003,TXN-0000007,CUST-06772,confirmed_fraud,2026-09-28T16:22:16.000Z,card_not_present,17932.68
CASE-0000004,TXN-0000067,CUST-05578,confirmed_fraud,2026-09-24T00:54:43.000Z,card_not_present,1132.09
CASE-0000005,TXN-0000067,CUST-05578,confirmed_fraud,2026-08-29T23:26:53.000Z,counterfeit,1887.84


investigation_outcome,count
confirmed_fraud,30001
false_positive,14708
suspicious_not_confirmed,5291


avg_recovery_amount,recovery_percentiles
373.35,"List(21.03, 747.11, 6522.72)"


In [0]:
customer_val = spark.table(CUSTOMER_TABLE)
merchant_val = spark.table(MERCHANT_TABLE)
transaction_val = spark.table(TRANSACTION_TABLE)
case_val = spark.table(CASE_TABLE)

expected_counts = {
    "dim_customer": N_CUSTOMERS,
    "dim_merchant": N_MERCHANTS,
    "fact_transactions": N_TRANSACTIONS,
    "fact_prior_fraud_cases": N_CASES,
}
actual_counts = {
    "dim_customer": customer_val.count(),
    "dim_merchant": merchant_val.count(),
    "fact_transactions": transaction_val.count(),
    "fact_prior_fraud_cases": case_val.count(),
}
assert actual_counts == expected_counts, {"expected": expected_counts, "actual": actual_counts}
print({"row_counts": actual_counts})

customer_fk_missing = transaction_val.join(customer_val.select("customer_id"), on="customer_id", how="left_anti").count()
merchant_fk_missing = transaction_val.join(merchant_val.select("merchant_id"), on="merchant_id", how="left_anti").count()
case_txn_fk_missing = case_val.join(transaction_val.select("txn_id"), on="txn_id", how="left_anti").count()
case_customer_fk_missing = case_val.join(customer_val.select("customer_id"), on="customer_id", how="left_anti").count()
assert customer_fk_missing == 0 and merchant_fk_missing == 0 and case_txn_fk_missing == 0 and case_customer_fk_missing == 0, {
    "customer_fk_missing": customer_fk_missing,
    "merchant_fk_missing": merchant_fk_missing,
    "case_txn_fk_missing": case_txn_fk_missing,
    "case_customer_fk_missing": case_customer_fk_missing,
}
print({
    "customer_fk_missing": customer_fk_missing,
    "merchant_fk_missing": merchant_fk_missing,
    "case_txn_fk_missing": case_txn_fk_missing,
    "case_customer_fk_missing": case_customer_fk_missing,
})

validation_customer_window = Window.partitionBy("customer_id").orderBy("timestamp", "txn_id")
validation_device_window = Window.partitionBy("customer_id", "device_id")

validation_base = (
    transaction_val
    .join(customer_val.select("customer_id", "home_region", "risk_segment", "avg_monthly_spend"), on="customer_id", how="inner")
    .join(merchant_val.select("merchant_id", "risk_tier", "country"), on="merchant_id", how="inner")
    .withColumn("home_country", region_to_country(F.col("home_region")))
    .withColumn("cross_border", F.col("country") != F.col("home_country"))
    .withColumn("customer_txn_seq", F.row_number().over(validation_customer_window))
    .withColumn("first_seen_seq", F.min("customer_txn_seq").over(validation_device_window))
    .withColumn("new_device_online", (F.col("channel") == "online") & (F.col("customer_txn_seq") == F.col("first_seen_seq")))
    .withColumn("high_amount_crossborder", (F.col("amount") > 2000) & F.col("cross_border"))
    .withColumn("unusual_spend", F.col("amount") > F.col("avg_monthly_spend") * 3)
    .withColumn("crossborder_highrisk_combo", F.col("cross_border") & (F.col("risk_tier") == "high") & (F.col("risk_segment") == "high"))
)

cohort_stats = (
    validation_base.filter(F.col("high_amount_crossborder"))
        .agg(F.lit("high_amount_crossborder").alias("cohort"), F.count("*").alias("txn_count"), F.round(F.avg(F.col("is_fraud").cast("double")) * 100, 2).alias("fraud_rate_pct"))
    .unionByName(
        validation_base.filter(F.col("new_device_online"))
            .agg(F.lit("new_device_online").alias("cohort"), F.count("*").alias("txn_count"), F.round(F.avg(F.col("is_fraud").cast("double")) * 100, 2).alias("fraud_rate_pct"))
    )
    .unionByName(
        validation_base.filter(F.col("unusual_spend"))
            .agg(F.lit("unusual_spend").alias("cohort"), F.count("*").alias("txn_count"), F.round(F.avg(F.col("is_fraud").cast("double")) * 100, 2).alias("fraud_rate_pct"))
    )
    .unionByName(
        validation_base.filter(F.col("crossborder_highrisk_combo"))
            .agg(F.lit("crossborder_highrisk_combo").alias("cohort"), F.count("*").alias("txn_count"), F.round(F.avg(F.col("is_fraud").cast("double")) * 100, 2).alias("fraud_rate_pct"))
    )
    .unionByName(
        validation_base.filter(~(F.col("high_amount_crossborder") | F.col("new_device_online") | F.col("unusual_spend") | F.col("crossborder_highrisk_combo")))
            .agg(F.lit("normal_transactions").alias("cohort"), F.count("*").alias("txn_count"), F.round(F.avg(F.col("is_fraud").cast("double")) * 100, 2).alias("fraud_rate_pct"))
    )
)

overall_fraud = transaction_val.agg(F.round(F.avg(F.col("is_fraud").cast("double")) * 100, 2).alias("overall_fraud_rate_pct")).first()["overall_fraud_rate_pct"]
print({"overall_fraud_rate_pct": overall_fraud})

display(cohort_stats)
display(customer_val.groupBy("home_region", "risk_segment").count().orderBy("home_region", "risk_segment"))
display(merchant_val.groupBy("mcc_category", "country").count().orderBy("mcc_category", "country"))
display(transaction_val.groupBy("channel", "is_fraud").count().orderBy("channel", "is_fraud"))
display(case_val.groupBy("investigation_outcome", "fraud_type").count().orderBy("investigation_outcome", "fraud_type"))

{'row_counts': {'dim_customer': 10000, 'dim_merchant': 5000, 'fact_transactions': 500000, 'fact_prior_fraud_cases': 50000}}
{'customer_fk_missing': 0, 'merchant_fk_missing': 0, 'case_txn_fk_missing': 0, 'case_customer_fk_missing': 0}
{'overall_fraud_rate_pct': 2.3}


cohort,txn_count,fraud_rate_pct
high_amount_crossborder,5380,27.79
new_device_online,22743,16.46
unusual_spend,8682,24.59
crossborder_highrisk_combo,10938,31.5
normal_transactions,457884,0.5


home_region,risk_segment,count
APAC,high,226
APAC,low,863
APAC,medium,406
EU,high,320
EU,low,1207
EU,medium,508
LATAM,high,153
LATAM,low,591
LATAM,medium,264
US-East,high,438


mcc_category,country,count
dining,BR,79
dining,CA,55
dining,DE,85
dining,FR,27
dining,IN,41
dining,JP,92
dining,MX,40
dining,SG,29
dining,UK,137
dining,US,414


channel,is_fraud,count
card_present,false,319914
card_present,true,5209
online,false,168608
online,true,6269


investigation_outcome,fraud_type,count
confirmed_fraud,account_takeover,7614
confirmed_fraud,card_not_present,11275
confirmed_fraud,counterfeit,7749
confirmed_fraud,friendly_fraud,2839
confirmed_fraud,identity_theft,524
false_positive,null,14708
suspicious_not_confirmed,null,5291


## Generation Summary

This notebook generated four Delta tables under `workshop_guy_catalog`.`fe-bar-a0`: `dim_customer`, `dim_merchant`, `fact_transactions`, and `fact_prior_fraud_cases`. The transaction fact table was seeded with overlapping fraud signals for high-value cross-border spend, online activity on new devices, unusual spend relative to customer baselines, and high-risk cross-border merchant/customer combinations.

The validation cell checks exact row counts, referential integrity, fraud-rate cohorts, and key categorical distributions so the dataset can be used for fraud analytics, feature engineering, and downstream model training demos.